# Medical Image Diagnosis Assistant
### Multi-Label Chest X-ray Disease Classification with Explainability

**Dataset:** NIH ChestX-ray14 (224×224 resized) | **Model:** DenseNet121 | **Framework:** PyTorch + MONAI

---

## Cell 1 — Environment Setup

Installs all dependencies, verifies GPU, enables mixed precision, fixes random seeds,
and prints exact software versions for reproducibility.


In [ ]:
# numpy is left unpinned: Colab's default numpy 2.x works fine with
# monai>=1.5.0 (earlier monai versions require numpy<2.0 and will crash).
!pip install -q torch torchvision "monai>=1.5.0" grad-cam==1.5.4 torchmetrics==1.4.0 pandas scikit-learn matplotlib opencv-python-headless tensorboard

import os
import random
from dataclasses import dataclass, field

import torch
import numpy as np
import monai
from torch.amp import GradScaler

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, go to: "
        "Runtime > Change runtime type > Hardware accelerator > GPU (T4 or better), "
        "then re-run this cell."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

# Mixed precision: forward pass runs in float16, GradScaler prevents small
# float16 gradients from underflowing to zero during backprop.
SCALER = GradScaler(device="cuda")

RANDOM_SEED = 42


def set_global_seed(seed: int) -> None:
    """Fix random seeds across every library for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(RANDOM_SEED)


@dataclass
class Config:
    dataset_root: str = "/content/drive/MyDrive/chestxray14"
    images_dir: str = field(init=False)
    data_entry_csv: str = field(init=False)
    bbox_csv: str = field(init=False)
    train_val_list_txt: str = field(init=False)
    test_list_txt: str = field(init=False)

    output_root: str = "/content/drive/MyDrive/chestxray14_project/outputs"
    checkpoints_dir: str = field(init=False)
    gradcam_dir: str = field(init=False)
    metrics_dir: str = field(init=False)
    logs_dir: str = field(init=False)
    splits_dir: str = field(init=False)

    image_size: int = 224
    val_fraction: float = 0.2
    random_seed: int = 42

    batch_size: int = 64
    num_workers: int = 4
    frozen_epochs: int = 3
    finetune_epochs: int = 8
    lr_frozen: float = 1e-3
    lr_finetune: float = 1e-5
    weight_decay: float = 1e-5
    early_stopping_patience: int = 3
    lr_scheduler_patience: int = 2
    lr_scheduler_factor: float = 0.5

    threshold_default: float = 0.5

    def __post_init__(self):
        self.images_dir = self._resolve_images_dir(self.dataset_root)
        self.data_entry_csv = os.path.join(self.dataset_root, "Data_Entry_2017.csv")
        self.bbox_csv = os.path.join(self.dataset_root, "BBox_List_2017_Official_NIH.csv")
        self.train_val_list_txt = os.path.join(self.dataset_root, "train_val_list_NIH.txt")
        self.test_list_txt = os.path.join(self.dataset_root, "test_list_NIH.txt")

        self.checkpoints_dir = os.path.join(self.output_root, "checkpoints")
        self.gradcam_dir = os.path.join(self.output_root, "gradcam")
        self.metrics_dir = os.path.join(self.output_root, "metrics")
        self.logs_dir = os.path.join(self.output_root, "logs")
        self.splits_dir = os.path.join(self.output_root, "splits")

        for d in (self.output_root, self.checkpoints_dir, self.gradcam_dir,
                  self.metrics_dir, self.logs_dir, self.splits_dir):
            os.makedirs(d, exist_ok=True)

    @staticmethod
    def _resolve_images_dir(dataset_root: str) -> str:
        """Find the folder actually containing the .png files.

        Some Kaggle mirrors nest images as images-224/images-224/*.png
        instead of the expected images-224/*.png.
        """
        outer = os.path.join(dataset_root, "images-224")
        nested = os.path.join(outer, "images-224")

        def has_images(path):
            return os.path.isdir(path) and any(
                f.lower().endswith((".png", ".jpg", ".jpeg")) for f in os.listdir(path)
            )

        if has_images(outer):
            return outer
        if has_images(nested):
            return nested
        return outer


def verify_dataset_paths(cfg: "Config") -> None:
    """Raise a clear error if any required dataset file is missing."""
    required = {
        "images-224/ directory": cfg.images_dir,
        "Data_Entry_2017.csv": cfg.data_entry_csv,
        "BBox_List_2017_Official_NIH.csv": cfg.bbox_csv,
        "train_val_list_NIH.txt": cfg.train_val_list_txt,
        "test_list_NIH.txt": cfg.test_list_txt,
    }
    missing = [name for name, path in required.items() if not os.path.exists(path)]
    if missing:
        raise FileNotFoundError(
            "The following required dataset files/folders were not found:\n"
            + "\n".join(f"  - {m}" for m in missing)
            + f"\n\nCheck that CONFIG.dataset_root ('{cfg.dataset_root}') "
              "points to the correct folder (run Cell 2 -- Dataset Acquisition "
              "-- if you have not already)."
        )
    num_images = len([f for f in os.listdir(cfg.images_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    print(f"Dataset paths verified. Images found in images-224/: {num_images}")


CONFIG = Config()

if os.path.exists(CONFIG.data_entry_csv):
    verify_dataset_paths(CONFIG)
else:
    print("Dataset not found yet at CONFIG.dataset_root -- run Cell 2 (Dataset Acquisition) next.")

print(f"GPU: {GPU_NAME} ({GPU_VRAM_GB:.1f} GB) | PyTorch {torch.__version__} | "
      f"MONAI {monai.__version__} | seed={RANDOM_SEED}")
print(f"Dataset root: {CONFIG.dataset_root}")
print(f"Images dir  : {CONFIG.images_dir}")


---
## Cell 2 — Dataset Acquisition (Kaggle)

Downloads the NIH ChestX-ray14 (224x224 resized) dataset directly from Kaggle
using `kagglehub`, copies it into Google Drive so it persists across Colab
runtime restarts, then stages the image folder onto Colab's local disk for
fast training access (reading 100k+ small files directly from a mounted
Drive is extremely slow -- see the "STAGE IMAGES ON LOCAL DISK" step below
for details).

**One-time setup required before running this cell:**
1. Get a Kaggle API token: on kaggle.com, go to your account settings →
   API section → "Create New API Token". This downloads a `kaggle.json`
   file containing your `username` and `key`.
2. In this Colab notebook, click the **key icon** in the left sidebar
   ("Secrets"), and add two secrets:
   - `KAGGLE_USERNAME` = the `username` value from your `kaggle.json`
   - `KAGGLE_KEY` = the `key` value from your `kaggle.json`
3. Toggle "Notebook access" on for both secrets.

This is the current Colab-recommended way to store credentials (safer than
uploading `kaggle.json` into the session, since Secrets persist across
sessions without re-uploading).


In [ ]:
!pip install -q kagglehub

import os
import shutil
import time

import kagglehub

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as secrets_error:
    kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(kaggle_json_path):
        raise RuntimeError(
            "Could not find Kaggle credentials. Either:\n"
            "  (a) Add KAGGLE_USERNAME and KAGGLE_KEY as Colab Secrets "
            "(key icon in the left sidebar), and enable notebook access, or\n"
            "  (b) Upload your kaggle.json to ~/.kaggle/kaggle.json manually.\n"
            f"Original error: {secrets_error}"
        )

KAGGLE_DATASET_HANDLE = "khanfashee/nih-chest-x-ray-14-224x224-resized"

print(f"Downloading {KAGGLE_DATASET_HANDLE} from Kaggle...")
kaggle_cache_path = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)

DRIVE_DATASET_ROOT = "/content/drive/MyDrive/chestxray14"
drive_marker_file = os.path.join(DRIVE_DATASET_ROOT, "Data_Entry_2017.csv")

if not os.path.exists(drive_marker_file):
    print(f"Copying to Drive ({DRIVE_DATASET_ROOT}) for persistence...")
    os.makedirs(DRIVE_DATASET_ROOT, exist_ok=True)
    shutil.copytree(kaggle_cache_path, DRIVE_DATASET_ROOT, dirs_exist_ok=True)

CONFIG.dataset_root = DRIVE_DATASET_ROOT
CONFIG.__post_init__()
verify_dataset_paths(CONFIG)

# Reading 100k+ files directly from mounted Drive is far slower than local
# disk (each read is a network round trip). Stage them locally once per
# session for fast DataLoader access -- CSVs stay on Drive since they're
# small and read only a handful of times.
LOCAL_IMAGES_DIR = "/content/images-224-local"


def stage_images_locally(source_images_dir: str, local_images_dir: str) -> str:
    """Copy the resolved images directory to local disk if not already staged."""
    marker_file = os.path.join(local_images_dir, ".staging_complete")
    if os.path.exists(marker_file):
        return local_images_dir

    print(f"Staging images to local disk ({local_images_dir})...")
    start_time = time.time()
    os.makedirs(local_images_dir, exist_ok=True)
    shutil.copytree(source_images_dir, local_images_dir, dirs_exist_ok=True)
    with open(marker_file, "w") as f:
        f.write("complete")
    print(f"Done in {time.time() - start_time:.0f}s.")
    return local_images_dir


CONFIG.images_dir = stage_images_locally(CONFIG.images_dir, LOCAL_IMAGES_DIR)
print(f"Images dir: {CONFIG.images_dir}")


---
## Cell 3 — Data Exploration

Loads `Data_Entry_2017.csv` (via `CONFIG.data_entry_csv`, set in Cell 2), parses
the multi-label `Finding Labels` column into a binary multi-hot matrix, reports
dataset-wide statistics (image count, patient count, class distribution), plots
class frequency, and sanity-checks a few sample images against their labels
and against `CONFIG.images_dir`.

Uses the single `CONFIG` object from Cell 1/2 rather than re-declaring its own
paths, so a change to `CONFIG.dataset_root` anywhere upstream propagates here
automatically.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IMAGES_DIR = CONFIG.images_dir
DATA_ENTRY_CSV = CONFIG.data_entry_csv

if not os.path.exists(DATA_ENTRY_CSV):
    raise FileNotFoundError(
        f"Could not find Data_Entry_2017.csv at {DATA_ENTRY_CSV}. "
        "Make sure Cell 2 (Dataset Acquisition) ran successfully first."
    )

data_entry_df = pd.read_csv(DATA_ENTRY_CSV)
print(f"Loaded {len(data_entry_df)} rows.")
data_entry_df.head()


def parse_all_labels(df: pd.DataFrame, label_column: str = "Finding Labels") -> list:
    """Return the sorted list of unique disease labels present in the dataset."""
    unique_labels = set()
    for label_string in df[label_column]:
        unique_labels.update(label_string.split("|"))
    return sorted(unique_labels)


ALL_LABELS = parse_all_labels(data_entry_df)
DISEASE_LABELS = [lbl for lbl in ALL_LABELS if lbl != "No Finding"]
NUM_CLASSES = len(DISEASE_LABELS)
print(f"{NUM_CLASSES} disease classes: {DISEASE_LABELS}")

for label in DISEASE_LABELS:
    data_entry_df[label] = data_entry_df["Finding Labels"].apply(
        lambda finding_string, lbl=label: 1 if lbl in finding_string.split("|") else 0
    )

data_entry_df["No Finding Flag"] = data_entry_df["Finding Labels"].apply(
    lambda s: 1 if s == "No Finding" else 0
)

num_images = len(data_entry_df)
num_patients = data_entry_df["Patient ID"].nunique()
avg_images_per_patient = num_images / num_patients
num_healthy = int(data_entry_df["No Finding Flag"].sum())
num_with_findings = num_images - num_healthy
avg_labels_per_image = data_entry_df[DISEASE_LABELS].sum(axis=1).mean()

print(f"\n{num_images} images, {num_patients} patients ({avg_images_per_patient:.2f} avg per patient)")
print(f"No Finding: {num_healthy} ({100*num_healthy/num_images:.1f}%)  "
      f">=1 finding: {num_with_findings} ({100*num_with_findings/num_images:.1f}%)  "
      f"avg findings/diseased image: {avg_labels_per_image:.2f}")

class_counts = data_entry_df[DISEASE_LABELS].sum().sort_values(ascending=False)
class_percentages = (class_counts / num_images * 100).round(2)

class_distribution_df = pd.DataFrame({
    "Class": class_counts.index,
    "Positive Count": class_counts.values,
    "Positive %": class_percentages.values,
})
print("\nClass distribution:")
print(class_distribution_df.to_string(index=False))

# Motivates the pos_weight decision in Cell 7.
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nMost/least frequent class imbalance ratio: {imbalance_ratio:.1f}x")

fig, ax = plt.subplots(figsize=(10, 5))
class_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Class Distribution -- ChestX-ray14 Disease Labels", fontsize=13)
ax.set_ylabel("Number of Positive Images")
ax.set_xlabel("Disease Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


def sanity_check_samples(df: pd.DataFrame, images_dir: str, num_samples: int = 4, seed: int = 42) -> None:
    """Display random sample images with their labels; verify files exist on disk."""
    sample_df = df.sample(n=min(num_samples, len(df)), random_state=seed)
    missing_files = []
    fig, axes = plt.subplots(1, len(sample_df), figsize=(4 * len(sample_df), 4))
    if len(sample_df) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        image_path = os.path.join(images_dir, row["Image Index"])
        if not os.path.exists(image_path):
            missing_files.append(image_path)
            ax.set_title(f"MISSING FILE\n{row['Image Index']}", fontsize=9, color="red")
            ax.axis("off")
            continue
        image = plt.imread(image_path)
        ax.imshow(image, cmap="gray")
        ax.set_title(f"{row['Finding Labels']}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    if missing_files:
        raise FileNotFoundError(
            f"{len(missing_files)} sample image(s) referenced in the CSV were "
            f"not found in {images_dir}:\n" + "\n".join(missing_files)
        )


sanity_check_samples(data_entry_df, IMAGES_DIR, num_samples=4, seed=42)


---
## Cell 4 — Label Engineering & Patient-Disjoint Split

Builds the full multi-hot label matrix, applies the official NIH `train_val` /
`test` split from `train_val_list_NIH.txt` / `test_list_NIH.txt`, then splits
`train_val` into train/val **by Patient ID** (80/20, patient-disjoint) so no
single patient's images appear in both train and val.

This cell is self-contained -- it reloads and re-parses `Data_Entry_2017.csv`
independently rather than relying on variables left over from Cell 3, so it
can be run on its own in a fresh session (as long as Cell 1 + Cell 2 have run
first, since it needs `CONFIG`).

Outputs three manifest CSVs to `CONFIG.splits_dir`: `train.csv`, `val.csv`,
`test.csv` -- each containing `Image Index`, `Patient ID`, `Finding Labels`,
and one binary column per disease class.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_entry_df = pd.read_csv(CONFIG.data_entry_csv)


def parse_all_labels(df: pd.DataFrame, label_column: str = "Finding Labels") -> list:
    """Return the sorted list of unique disease labels present in the dataset."""
    unique_labels = set()
    for label_string in df[label_column]:
        unique_labels.update(label_string.split("|"))
    return sorted(unique_labels)


ALL_LABELS = parse_all_labels(data_entry_df)
DISEASE_LABELS = [lbl for lbl in ALL_LABELS if lbl != "No Finding"]
NUM_CLASSES = len(DISEASE_LABELS)

for label in DISEASE_LABELS:
    data_entry_df[label] = data_entry_df["Finding Labels"].apply(
        lambda finding_string, lbl=label: 1 if lbl in finding_string.split("|") else 0
    )

print(f"Loaded {len(data_entry_df)} images, {NUM_CLASSES} disease classes.")


def load_image_list(txt_path: str) -> set:
    """Read a newline-delimited list of image filenames into a set."""
    with open(txt_path, "r") as f:
        return set(line.strip() for line in f if line.strip())


train_val_image_names = load_image_list(CONFIG.train_val_list_txt)
test_image_names = load_image_list(CONFIG.test_list_txt)

print(f"train_val_list_NIH.txt : {len(train_val_image_names)} images")
print(f"test_list_NIH.txt      : {len(test_image_names)} images")

list_overlap = train_val_image_names & test_image_names
if list_overlap:
    raise ValueError(
        f"{len(list_overlap)} image(s) appear in BOTH train_val_list_NIH.txt "
        f"and test_list_NIH.txt. This should never happen -- check the "
        f"source files. Example overlapping filenames: {list(list_overlap)[:5]}"
    )

train_val_df = data_entry_df[data_entry_df["Image Index"].isin(train_val_image_names)].reset_index(drop=True)
test_df = data_entry_df[data_entry_df["Image Index"].isin(test_image_names)].reset_index(drop=True)

matched_train_val = len(train_val_df)
matched_test = len(test_df)
print(f"Matched in CSV -- train_val: {matched_train_val}/{len(train_val_image_names)}, "
      f"test: {matched_test}/{len(test_image_names)}")

if matched_train_val < len(train_val_image_names) or matched_test < len(test_image_names):
    print(
        "WARNING: not every filename in the official split lists was found "
        "in Data_Entry_2017.csv. This can happen if the CSV and split lists "
        "come from slightly different dataset versions -- proceeding with "
        "only the matched images."
    )


def patient_disjoint_split(df: pd.DataFrame, val_fraction: float, seed: int) -> tuple:
    """Split by Patient ID (not row) so no patient appears in both train and val."""
    unique_patient_ids = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    shuffled_patient_ids = unique_patient_ids.copy()
    rng.shuffle(shuffled_patient_ids)

    num_val_patients = int(round(len(shuffled_patient_ids) * val_fraction))
    val_patient_ids = set(shuffled_patient_ids[:num_val_patients])
    train_patient_ids = set(shuffled_patient_ids[num_val_patients:])

    train_split_df = df[df["Patient ID"].isin(train_patient_ids)].reset_index(drop=True)
    val_split_df = df[df["Patient ID"].isin(val_patient_ids)].reset_index(drop=True)
    return train_split_df, val_split_df


train_df, val_df = patient_disjoint_split(
    train_val_df, val_fraction=CONFIG.val_fraction, seed=CONFIG.random_seed
)

# Train vs val is under our control (produced above) -- a violation here is
# a real bug and stays a hard assertion. Train/val vs test depends on the
# official NIH split files being patient-disjoint (documented to be true
# for the canonical dataset, but not something our code controls), so that
# case only warns rather than crashing the whole pipeline.
train_patient_ids = set(train_df["Patient ID"])
val_patient_ids = set(val_df["Patient ID"])
test_patient_ids = set(test_df["Patient ID"])

assert train_patient_ids.isdisjoint(val_patient_ids), (
    "Patient leakage detected between train and val -- this indicates a bug "
    "in patient_disjoint_split() and must be fixed before proceeding."
)

train_test_overlap = train_patient_ids & test_patient_ids
val_test_overlap = val_patient_ids & test_patient_ids
if train_test_overlap or val_test_overlap:
    print(
        "WARNING: the official train_val_list_NIH.txt / test_list_NIH.txt "
        "files are NOT patient-disjoint for this dataset copy -- "
        f"{len(train_test_overlap)} patient(s) overlap train/test, "
        f"{len(val_test_overlap)} patient(s) overlap val/test. This means "
        "some test-set images may share a patient with training images, "
        "which can inflate test metrics. This is a property of the source "
        "data files, not of this notebook's code -- proceeding anyway, but "
        "keep this in mind when interpreting test-set results."
    )
else:
    print("Patient-disjointness verified: no leakage across train/val/test.")

split_summary_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Images": [len(train_df), len(val_df), len(test_df)],
    "Patients": [len(train_patient_ids), len(val_patient_ids), len(test_patient_ids)],
})
split_summary_df["Image %"] = (
    split_summary_df["Images"] / split_summary_df["Images"].sum() * 100
).round(1)
print("\nSplit summary:")
print(split_summary_df.to_string(index=False))

class_dist_by_split = pd.DataFrame({
    "Train %": (train_df[DISEASE_LABELS].sum() / max(len(train_df), 1) * 100),
    "Val %": (val_df[DISEASE_LABELS].sum() / max(len(val_df), 1) * 100),
    "Test %": (test_df[DISEASE_LABELS].sum() / max(len(test_df), 1) * 100),
}).round(2)
print("\nPer-class positive rate (%) by split:")
print(class_dist_by_split.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
class_dist_by_split.plot(kind="bar", ax=ax)
ax.set_title("Class Positive Rate by Split (Train / Val / Test)", fontsize=13)
ax.set_ylabel("% of images positive for class")
ax.set_xlabel("Disease Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

manifest_columns = ["Image Index", "Patient ID", "Finding Labels"] + DISEASE_LABELS

train_csv_path = os.path.join(CONFIG.splits_dir, "train.csv")
val_csv_path = os.path.join(CONFIG.splits_dir, "val.csv")
test_csv_path = os.path.join(CONFIG.splits_dir, "test.csv")

train_df[manifest_columns].to_csv(train_csv_path, index=False)
val_df[manifest_columns].to_csv(val_csv_path, index=False)
test_df[manifest_columns].to_csv(test_csv_path, index=False)

labels_json_path = os.path.join(CONFIG.splits_dir, "disease_labels.json")
with open(labels_json_path, "w") as f:
    json.dump(DISEASE_LABELS, f, indent=2)

print(f"\nSaved to {CONFIG.splits_dir}: train ({len(train_df)}), "
      f"val ({len(val_df)}), test ({len(test_df)}), {NUM_CLASSES} classes.")


---
## Cell 5 — Dataset & MONAI Transforms

Builds a PyTorch `Dataset` that reads from the split manifests saved in
Cell 4 (`train.csv` / `val.csv` / `test.csv`) and loads the actual image
files from `CONFIG.images_dir`. Defines two MONAI transform pipelines: a
training pipeline with medical-image augmentation (flip, small rotation,
contrast jitter, mild noise), and a deterministic pipeline for val/test.
Wraps both in `DataLoader`s, then pulls one real batch and displays a few
augmented images to visually confirm the pipeline before building the model
around it.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from monai.transforms import (
    Compose, EnsureChannelFirst, ScaleIntensity, RepeatChannel, Resize,
    RandFlip, RandRotate, RandAdjustContrast, RandGaussianNoise,
    NormalizeIntensity, ToTensor,
)

labels_json_path = os.path.join(CONFIG.splits_dir, "disease_labels.json")
if not os.path.exists(labels_json_path):
    raise FileNotFoundError(
        f"Could not find {labels_json_path}. Run Cell 4 (Label Engineering & "
        "Split) first -- it produces this file along with train/val/test.csv."
    )
with open(labels_json_path, "r") as f:
    DISEASE_LABELS = json.load(f)
NUM_CLASSES = len(DISEASE_LABELS)
print(f"{NUM_CLASSES} disease classes loaded.")

# ImageNet stats -- required since the DenseNet121 backbone (Cell 6) is
# ImageNet-pretrained and expects inputs normalized the same way.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_shared_prefix = [
    EnsureChannelFirst(channel_dim="no_channel"),
    ScaleIntensity(minv=0.0, maxv=1.0),
    RepeatChannel(repeats=3),
    Resize(spatial_size=(CONFIG.image_size, CONFIG.image_size)),
]
_shared_suffix = [
    NormalizeIntensity(subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    ToTensor(),
]

# Horizontal flip only -- vertical flip would invert anatomy in a way that
# never occurs in real scans. Rotation kept small (~10 degrees) since large
# rotations are physically implausible for a chest X-ray.
train_transforms = Compose(_shared_prefix + [
    RandFlip(prob=0.5, spatial_axis=1),
    RandRotate(range_x=0.175, prob=0.5, padding_mode="zeros"),
    RandAdjustContrast(prob=0.5, gamma=(0.8, 1.2)),
    RandGaussianNoise(prob=0.2, mean=0.0, std=0.01),
] + _shared_suffix)

eval_transforms = Compose(_shared_prefix + _shared_suffix)


class ChestXrayDataset(Dataset):
    """Multi-label chest X-ray dataset reading from a split manifest CSV."""

    def __init__(self, manifest_csv_path: str, images_dir: str,
                 disease_labels: list, transform: Compose):
        self.manifest_df = pd.read_csv(manifest_csv_path)
        self.images_dir = images_dir
        self.disease_labels = disease_labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.manifest_df)

    def __getitem__(self, index: int):
        row = self.manifest_df.iloc[index]
        image_path = os.path.join(self.images_dir, row["Image Index"])

        if not os.path.exists(image_path):
            raise FileNotFoundError(
                f"Image file not found: {image_path}. This manifest row "
                f"expects it to exist -- check CONFIG.images_dir is correct."
            )

        # Some images in this dataset are stored as RGB despite being
        # inherently single-channel X-rays -- force grayscale for consistency.
        pil_image = Image.open(image_path).convert("L")
        image_array = np.array(pil_image)

        transformed_image = self.transform(image_array)

        label_values = row[self.disease_labels].values.astype(np.float32)
        label_tensor = torch.from_numpy(label_values)

        return transformed_image, label_tensor


train_dataset = ChestXrayDataset(
    manifest_csv_path=os.path.join(CONFIG.splits_dir, "train.csv"),
    images_dir=CONFIG.images_dir,
    disease_labels=DISEASE_LABELS,
    transform=train_transforms,
)
val_dataset = ChestXrayDataset(
    manifest_csv_path=os.path.join(CONFIG.splits_dir, "val.csv"),
    images_dir=CONFIG.images_dir,
    disease_labels=DISEASE_LABELS,
    transform=eval_transforms,
)
test_dataset = ChestXrayDataset(
    manifest_csv_path=os.path.join(CONFIG.splits_dir, "test.csv"),
    images_dir=CONFIG.images_dir,
    disease_labels=DISEASE_LABELS,
    transform=eval_transforms,
)

print(f"Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)} images")

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=True,
    num_workers=CONFIG.num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=True,
)

print(f"\nBatches per epoch -- train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")


def denormalize_for_display(image_tensor: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalization for display -- returns (H, W, 3) in [0, 1]."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    denormalized = image_tensor * std + mean
    denormalized = denormalized.clamp(0, 1)
    return denormalized.permute(1, 2, 0).numpy()


sample_images, sample_labels = next(iter(train_loader))
print(f"\nBatch shapes: images {tuple(sample_images.shape)}, labels {tuple(sample_labels.shape)}")

num_to_show = min(4, sample_images.shape[0])
fig, axes = plt.subplots(1, num_to_show, figsize=(4 * num_to_show, 4))
if num_to_show == 1:
    axes = [axes]

for i in range(num_to_show):
    display_image = denormalize_for_display(sample_images[i])
    positive_classes = [
        DISEASE_LABELS[j] for j in range(NUM_CLASSES) if sample_labels[i, j] == 1
    ]
    title = ", ".join(positive_classes) if positive_classes else "No Finding"
    axes[i].imshow(display_image)
    axes[i].set_title(title, fontsize=9)
    axes[i].axis("off")

plt.suptitle("Sample Augmented Training Batch", fontsize=12)
plt.tight_layout()
plt.show()
